# 🎫 Support Ticket Classifier — Jupyter Notebook
**Task 2 | NLP Classification Project**

This notebook walks through the full pipeline:
1. Dataset generation & EDA
2. Text preprocessing
3. Model training (TF-IDF + Logistic Regression)
4. Evaluation & visualizations
5. Live predictions

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from src.classifier import TicketClassifier, TextPreprocessor, assign_priority
from src.dataset import generate_dataset

print('✅ Imports successful')

## 1. Generate & Explore Dataset

In [ ]:
df = generate_dataset(samples_per_category=30)
print(f'Total tickets: {len(df)}')
df.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Category distribution
df['category'].value_counts().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Tickets per Category')
axes[0].set_xlabel('Count')

# Priority distribution
colors = ['#e74c3c', '#f39c12', '#2ecc71']
df['priority'].value_counts().plot(kind='pie', ax=axes[1], colors=colors,
                                    autopct='%1.1f%%', startangle=90)
axes[1].set_title('Priority Distribution')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 2. Text Preprocessing Demo

In [ ]:
prep = TextPreprocessor()
sample = "URGENT! My application keeps crashing & I can't login. Please help ASAP!"
print('Original  :', sample)
print('Cleaned   :', prep.clean(sample))
print('Processed :', prep.preprocess(sample))

## 3. Train Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['ticket_text'].tolist(), df['category'].tolist(),
    test_size=0.2, random_state=42, stratify=df['category']
)

clf = TicketClassifier()
clf.train(X_train, y_train)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

## 4. Evaluate

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = clf.evaluate(X_test, y_test)

labels = sorted(df['category'].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Live Predictions

In [ ]:
test_tickets = [
    "URGENT: Production server is down, no one can access the system!",
    "I would love a dark mode option in your mobile app.",
    "I was charged twice this month, please issue a refund.",
    "How do I change my account email address?",
    "Your support team took 3 weeks to respond, very disappointed.",
]

for ticket in test_tickets:
    r = clf.predict(ticket)
    print(f'Ticket   : {ticket[:60]}...')
    print(f'Category : {r["category"]}  |  Priority: {r["priority"]}  |  Confidence: {r["confidence"]}')
    print()